# Conditional SE(3) Diffusion Training (STAR-MD)

Trains a **conditional** STAR-MD score network on a single protein MD
trajectory using the SinFusion single-trajectory protocol with
Diffusion Forcing.

**What you get:** A model that autoregressively generates MD trajectories
frame-by-frame, conditioned on previous frames.

| Step | Description |
|------|-------------|
| 1 | Mount Google Drive (persistent storage) |
| 2 | Clone repo & init submodules |
| 3 | Configure protein & training |
| 4 | Download & preprocess trajectory |
| 5 | Train (STAR-MD with Diffusion Forcing) |
| 6 | Generate trajectory (autoregressive rollout) |

## Step 1: Environment Setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
    print('Running in Google Colab')
except ImportError:
    IN_COLAB = False
    print('Running locally')

In [ ]:
import os

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/protein_data/data', exist_ok=True)
    print('Drive mounted')

In [ ]:
# Install dependencies

# Core (required for training)
!pip install -q pytorch-lightning omegaconf biopython dm-tree ml-collections

# Evaluation: mdtraj, deeptime (TICA), statsmodels (autocorrelation)
!pip install -q 'numpy<2' mdtraj statsmodels deeptime

# Optional: pyemma (uses MDGen's full analysis script; falls back to deeptime if unavailable)
!pip install -q pyemma 2>/dev/null || echo 'pyemma not installed — will use deeptime fallback for TICA'
# MDGen baseline + optional
!pip install -q huggingface-hub py3Dmol optuna 2>/dev/null || true

# Verify
!python -c 'import mdtraj; print(f"mdtraj {mdtraj.__version__} OK")'
!python -c 'import deeptime; print(f"deeptime {deeptime.__version__} OK")'
!python -c 'import numpy; print(f"numpy {numpy.__version__}")'

print('Dependencies installed')


## Step 2: Clone Repository & Init Submodules

In [ ]:
import os, subprocess, shutil

REPO_URL = 'https://github.com/JiwonJJeong/winter-gen-pproject.git'

if IN_COLAB:
    os.chdir('/content')
    # Fresh clone each time to avoid dirty-state conflicts
    if os.path.exists('winter-gen-pproject'):
        shutil.rmtree('winter-gen-pproject')
    subprocess.run(['git', 'clone', '--recurse-submodules', REPO_URL], check=True)
    os.chdir('winter-gen-pproject')
    # Symlink data/ to Drive for persistence
    if not os.path.islink('data'):
        os.symlink('/content/drive/MyDrive/protein_data/data', 'data')
        print('data/ -> /content/drive/MyDrive/protein_data/data')
    print('Code ready')
else:
    assert os.path.isdir('gen_model'), 'Run from repo root'
    print(f'Local repo: {os.path.abspath(".")}')


In [ ]:
from gen_model.viz import viz_trajectory_pt, viz_trajectory_pdb
print('Visualization utilities loaded')


## Step 3: Configuration

Edit the variables below for your protein and training preferences.

In [ ]:
# ---- Protein (single-trajectory, SinFusion-style) ----
PROTEIN   = '4o66_C'   # Protein name (as in atlas.csv)
REPLICA   = '1'        # Which replica (1, 2, or 3)

# ---- Generalization test (ill-posed but informative) ----
EVAL_PROTEIN = '6ono_C'  # 76-residue protein from MDGen/STAR-MD test set (never in any training split)

# ---- Training ----
MAX_STEPS  = 50_000    # 50k for Colab; 200k for full training
BATCH_SIZE = 1         # SinFusion default for single-trajectory
LR         = 1e-4
GRAD_CLIP  = 1.0
EMA_DECAY  = 0.999

# ---- STAR-MD / Diffusion Forcing ----
NUM_FRAMES = 8         # Window length L (paper training: 8)
MIN_T      = 0.01
MAX_T      = 0.1
NS_PER_FRAME = 0.1
CURRICULUM = True
SPATIAL_SIGMA = 15.0   # Gaussian spatial bias in ST attention (Angstroms)

# ---- Optional steps (set False to skip when using Run All) ----
RUN_ABLATION      = False  # Step 9: local spatial attention ablation
RUN_MDGEN_TRAIN   = True   # Steps 12-13: train MDGen baselines

# ---- Paths (Drive on Colab, local otherwise) ----
DRIVE_BASE = '/content/drive/MyDrive/protein_data' if IN_COLAB else '.'
DATA_DIR   = f'{DRIVE_BASE}/data'
SAVE_DIR   = f'{DRIVE_BASE}/checkpoints/conditional/{PROTEIN}'
OUT_DIR    = f'{DRIVE_BASE}/outputs/conditional/{PROTEIN}'

print(f'Train protein : {PROTEIN}_R{REPLICA}')
print(f'Eval protein  : {EVAL_PROTEIN}')
print(f'Steps         : {MAX_STEPS:,}')
print(f'Window (L)    : {NUM_FRAMES}')
print(f'Save to       : {SAVE_DIR}')
print(f'Outputs to    : {OUT_DIR}')


## Step 4: Download & Prepare Data

In [ ]:
import os

if IN_COLAB and not os.path.islink('data'):
    raise RuntimeError('data/ not linked to Drive. Run Step 1 first.')

# Download training protein
!PYTHONPATH=. python scripts/download_and_prep.py {PROTEIN} --data_dir {DATA_DIR} --out_dir {DATA_DIR} --suffix _latent

# Download evaluation protein (for generalization test)
!PYTHONPATH=. python scripts/download_and_prep.py {EVAL_PROTEIN} --data_dir {DATA_DIR} --out_dir {DATA_DIR} --suffix _latent

for prot in [PROTEIN, EVAL_PROTEIN]:
    traj_path = f'{DATA_DIR}/{prot}/{prot}_R1_latent.npy'
    if os.path.exists(traj_path):
        import numpy as np
        arr = np.load(traj_path)
        print(f'  {prot}: {traj_path}  shape={arr.shape}')
    else:
        print(f'  ERROR: {traj_path} not found')


## Step 5: Train (STAR-MD + Diffusion Forcing)

Runs `gen_model/train_conditional.py` with:
- STAR-MD spatio-temporal attention (block-causal, 2D-RoPE, AdaLN)
- Diffusion Forcing: all L frames noised at same tau, loss at all positions
- SinFusion curriculum for delta_t (gradually increasing temporal range)
- EMA, gradient clipping, cosine LR, mixed precision

In [ ]:
curriculum_flag = '--curriculum' if CURRICULUM else '--no_curriculum'

!PYTHONPATH=. python gen_model/train_conditional.py \
    --protein {PROTEIN} \
    --replica {REPLICA} \
    --data_dir {DATA_DIR} \
    --batch_size {BATCH_SIZE} \
    --max_steps {MAX_STEPS} \
    --lr {LR} \
    --grad_clip {GRAD_CLIP} \
    --ema_decay {EMA_DECAY} \
    --num_frames {NUM_FRAMES} \
    --ns_per_stored_frame {NS_PER_FRAME} \
    --min_t {MIN_T} \
    --max_t {MAX_T} \
    {curriculum_flag} \
    --save_dir {SAVE_DIR} \
    --spatial_sigma {SPATIAL_SIGMA} \
    --num_blocks 4 \
    --num_workers 2

## Step 6: Generate Trajectory (Training Protein)

In [ ]:
import glob

TOTAL_FRAMES = 250
DELTA_T      = 0.1
N_STEPS      = 200

ckpts = sorted(glob.glob(f'{SAVE_DIR}/cond-*.ckpt'))
best_ckpt = ckpts[-1] if ckpts else f'{SAVE_DIR}/last.ckpt'
print(f'Checkpoint: {best_ckpt}')

!PYTHONPATH=. python gen_model/inference_conditional.py \
    --checkpoint {best_ckpt} \
    --data_dir {DATA_DIR} \
    --protein {PROTEIN} \
    --total_frames {TOTAL_FRAMES} \
    --num_frames {NUM_FRAMES} \
    --delta_t {DELTA_T} \
    --n_steps {N_STEPS} \
    --output {DRIVE_BASE}/outputs/conditional/{PROTEIN}/traj.pt


## Step 7: Evaluate (Training Protein)

Compare generated trajectory against reference MD (all 3 replicas).

In [ ]:
!PYTHONPATH=. python gen_model/evaluate.py \
    --ref_npy {DATA_DIR}/{PROTEIN}/{PROTEIN}_R1_latent.npy \
              {DATA_DIR}/{PROTEIN}/{PROTEIN}_R2_latent.npy \
              {DATA_DIR}/{PROTEIN}/{PROTEIN}_R3_latent.npy \
    --gen_traj outputs/conditional/{PROTEIN}/traj.pt \
    --protein {PROTEIN} --mode conditional --ref_mode all \
    --out_dir {OUT_DIR}/../eval/conditional/{PROTEIN}

# Display the evaluation plots (generated by MDGen analysis)
eval_pdf = f'{DRIVE_BASE}/outputs/eval/conditional/' + PROTEIN + '/gen/' + PROTEIN + '.pdf'
if os.path.exists(eval_pdf):
    from IPython.display import display, IFrame
    display(IFrame(eval_pdf, width=800, height=600))
else:
    print(f'No evaluation PDF found at {eval_pdf}')


## Step 8: Visualize (Training Protein)

In [ ]:
viz_trajectory_pt(f'{DRIVE_BASE}/outputs/conditional/{PROTEIN}/traj.pt',
                  f'{PROTEIN} (training protein)')


## Step 9: Ablation — Local Spatial Attention

Re-train with `--spatial_sigma 15.0` (SinFusion-inspired local receptive
field in ST attention) and compare against the global attention model
from Step 7.

The Gaussian bias `-(d/sigma)^2` on CA-CA distance means:
- <15A: full attention
- ~30A: strongly attenuated
- >45A: nearly zero

**Hypothesis:** Local attention should reduce overfitting on the single
trajectory by preventing memorization of global pairwise relationships.

In [ ]:
if RUN_ABLATION:
    SAVE_DIR_GLOBAL = SAVE_DIR.replace('conditional/', 'conditional_global/')
    
    curriculum_flag = '--curriculum' if CURRICULUM else '--no_curriculum'
    
    !PYTHONPATH=. python gen_model/train_conditional.py \
        --protein {PROTEIN} \
        --replica {REPLICA} \
        --data_dir {DATA_DIR} \
        --batch_size {BATCH_SIZE} \
        --max_steps {MAX_STEPS} \
        --lr {LR} \
        --grad_clip {GRAD_CLIP} \
        --ema_decay {EMA_DECAY} \
        --num_frames {NUM_FRAMES} \
        --ns_per_stored_frame {NS_PER_FRAME} \
        --min_t {MIN_T} \
        --max_t {MAX_T} \
        {curriculum_flag} \
        --spatial_sigma 0.0 \
        --save_dir {SAVE_DIR_GLOBAL} \
        --num_blocks 4 \
    --num_workers 2


In [ ]:
if RUN_ABLATION:
    import glob
    
    # Generate with local-attention model
    ckpts_local = sorted(glob.glob(f'{SAVE_DIR_LOCAL}/cond-*.ckpt'))
    best_ckpt_local = ckpts_local[-1] if ckpts_local else f'{SAVE_DIR_LOCAL}/last.ckpt'
    print(f'Local attention checkpoint: {best_ckpt_local}')
    
    !PYTHONPATH=. python gen_model/inference_conditional.py \
        --checkpoint {best_ckpt_local} \
        --data_dir {DATA_DIR} \
        --protein {PROTEIN} \
        --total_frames {TOTAL_FRAMES} \
        --num_frames {NUM_FRAMES} \
        --delta_t {DELTA_T} \
        --n_steps {N_STEPS} \
        --st_num_heads 8 \
        --output {DRIVE_BASE}/outputs/conditional_global/{PROTEIN}/traj.pt
    
    # Evaluate
    !PYTHONPATH=. python gen_model/evaluate.py \
        --ref_npy {DATA_DIR}/{PROTEIN}/{PROTEIN}_R1_latent.npy \
                  {DATA_DIR}/{PROTEIN}/{PROTEIN}_R2_latent.npy \
                  {DATA_DIR}/{PROTEIN}/{PROTEIN}_R3_latent.npy \
        --gen_traj outputs/conditional_global/{PROTEIN}/traj.pt \
        --protein {PROTEIN} --mode conditional --ref_mode all \
        --out_dir {OUT_DIR}/../eval/conditional_global/{PROTEIN}
    
    # Display evaluation PDFs
    import glob as _glob
    from IPython.display import display, IFrame
    for _pdf in _glob.glob(f'{DRIVE_BASE}/outputs/**/eval*/**/*.pdf', recursive=True)[-3:]:
        if _pdf not in globals().get('_shown_pdfs', set()):
            print(f'\n{_pdf}:')
            display(IFrame(_pdf, width=800, height=600))
            _shown_pdfs = globals().get('_shown_pdfs', set()) | {_pdf}
    
    import os, torch
    _traj = f'{DRIVE_BASE}/outputs/conditional_global/{PROTEIN}/traj.pt'
    if os.path.exists(_traj):
        _t = torch.load(_traj, map_location='cpu')
        _ca = _t[:, :, 4:7].numpy()
        import matplotlib.pyplot as plt
    import os, torch
    _traj = f'{DRIVE_BASE}/outputs/conditional_global/{PROTEIN}/traj.pt'
    if os.path.exists(_traj):
        _t = torch.load(_traj, map_location='cpu')
        _ca = _t[:, :, 4:7].numpy()
        _title = f'{PROTEIN} ablation (local sigma) — {len(_ca)} frames'
        import matplotlib.pyplot as plt, numpy as np
    
    viz_trajectory_pt(f'{DRIVE_BASE}/outputs/conditional_global/{PROTEIN}/traj.pt',
                      f'{PROTEIN} ablation (local sigma)')


In [ ]:
if RUN_ABLATION:
    # Side-by-side comparison: global vs local attention
    import pickle, numpy as np
    
    results = {}
    for label, eval_dir in [
        ('Global (sigma=0)', f'{DRIVE_BASE}/outputs/eval/conditional/{PROTEIN}'),
        ('Global (sigma=0)', f'{DRIVE_BASE}/outputs/eval/conditional_global/{PROTEIN}'),
    ]:
        pkl = f'{eval_dir}/gen/eval_results.pkl'
        if os.path.exists(pkl):
            with open(pkl, 'rb') as f:
                data = pickle.load(f)
            if PROTEIN in data and 'JSD' in data[PROTEIN]:
                results[label] = data[PROTEIN]['JSD']
    
    if len(results) == 2:
        print(f'{"Metric":<45s}  {"Global":>8s}  {"Local":>8s}  {"Delta":>8s}')
        print('=' * 75)
        labels = list(results.keys())
        for key in sorted(results[labels[0]].keys()):
            g = results[labels[0]][key]
            l = results[labels[1]][key]
            d = l - g
            better = '<' if d < 0 else '>'
            print(f'{key:<45s}  {g:8.4f}  {l:8.4f}  {d:+8.4f} {better}')
    else:
        print('Run both global (Step 7) and local (Step 9) evaluations first.')


In [ ]:
# ── Select best model for subsequent steps ─────────────────────────
# If Step 9 (ablation) was run, auto-selects the winner.
# If Step 9 was skipped, defaults to the Step 5 model (spatial sigma on).

import pickle, numpy as np

try:
    # Step 9 was run: compare local (default) vs global (ablation)
    if len(results) == 2:
        labels = list(results.keys())
        mean_jsd = {label: np.mean(list(jsd.values())) for label, jsd in results.items()}
        best_label = min(mean_jsd, key=mean_jsd.get)
        if 'Global' in best_label:
            best_ckpt = best_ckpt_global
            BEST_LABEL = 'global (sigma=0)'
        else:
            # best_ckpt already set in Step 6 (local, our default)
            BEST_LABEL = f'local (sigma={SPATIAL_SIGMA})'
        print(f'Ablation result: {best_label}')
        for l, v in mean_jsd.items():
            print(f'  {l}: mean JSD = {v:.4f}')
    else:
        BEST_LABEL = f'local (sigma={SPATIAL_SIGMA}, default)'
except NameError:
    # Step 9 was skipped — use the base model from Step 5 (spatial sigma on)
    BEST_LABEL = f'local (sigma={SPATIAL_SIGMA}, ablation skipped)'

print(f'\nUsing {BEST_LABEL} model for Steps 10+.')
print(f'Checkpoint: {best_ckpt}')


## Step 10: Generalization Test

Generate, evaluate, and visualize on `EVAL_PROTEIN` — a held-out protein
from the MDGen/STAR-MD test set (never in any training split).
Quantifies single-trajectory overfitting vs. generalization.

In [ ]:
print(f'Generalization test: {PROTEIN} (train) -> {EVAL_PROTEIN} (held-out)')
print(f'  {EVAL_PROTEIN} is from the MDGen/STAR-MD test set')

# Generate trajectory on held-out protein
!PYTHONPATH=. python gen_model/inference_conditional.py \
    --checkpoint {best_ckpt} \
    --data_dir {DATA_DIR} \
    --protein {EVAL_PROTEIN} \
    --total_frames {TOTAL_FRAMES} \
    --num_frames {NUM_FRAMES} \
    --delta_t {DELTA_T} \
    --n_steps {N_STEPS} \
    --output {DRIVE_BASE}/outputs/conditional/{EVAL_PROTEIN}/traj.pt

# Evaluate against all 3 replicas
!PYTHONPATH=. python gen_model/evaluate.py \
    --ref_npy {DATA_DIR}/{EVAL_PROTEIN}/{EVAL_PROTEIN}_R1_latent.npy \
              {DATA_DIR}/{EVAL_PROTEIN}/{EVAL_PROTEIN}_R2_latent.npy \
              {DATA_DIR}/{EVAL_PROTEIN}/{EVAL_PROTEIN}_R3_latent.npy \
    --gen_traj outputs/conditional/{EVAL_PROTEIN}/traj.pt \
    --protein {EVAL_PROTEIN} --mode conditional --ref_mode all \
    --out_dir {OUT_DIR}/../eval/conditional/{EVAL_PROTEIN}

# Display evaluation PDFs
import glob as _glob
from IPython.display import display, IFrame
for _pdf in _glob.glob(f'{DRIVE_BASE}/outputs/**/eval*/**/*.pdf', recursive=True)[-3:]:
    if _pdf not in globals().get('_shown_pdfs', set()):
        print(f'\n{_pdf}:')
        display(IFrame(_pdf, width=800, height=600))
        _shown_pdfs = globals().get('_shown_pdfs', set()) | {_pdf}


In [ ]:
import os, torch
traj_out = f'{DRIVE_BASE}/outputs/conditional/{EVAL_PROTEIN}/traj.pt'
if os.path.exists(traj_out):
    _t = torch.load(traj_out, map_location='cpu')
    _ca = _t[:, :, 4:7].numpy()
    _title = f'{EVAL_PROTEIN} (held-out protein) — {len(_ca)} frames'
    import matplotlib.pyplot as plt, numpy as np
else:
    print('No trajectory found.')


## Step 11: MDGen Baseline

Download the pre-trained MDGen ATLAS checkpoint from HuggingFace and run
inference on both proteins. MDGen is trained on 1266 ATLAS proteins
(multi-trajectory), so this provides an **upper-bound baseline** for
comparison against our single-trajectory model.

Model weights: https://huggingface.co/bjing-mit/mdgen

In [ ]:
import os, subprocess

# Download MDGen ATLAS checkpoint from HuggingFace
MDGEN_CKPT_DIR = f'{DRIVE_BASE}/checkpoints/mdgen'
MDGEN_CKPT = os.path.join(MDGEN_CKPT_DIR, 'atlas.ckpt')

if not os.path.exists(MDGEN_CKPT):
    os.makedirs(MDGEN_CKPT_DIR, exist_ok=True)
    print('Downloading MDGen ATLAS checkpoint from HuggingFace ...')
    subprocess.run([
        'huggingface-cli', 'download', 'bjing-mit/mdgen', 'atlas.ckpt',
        '--local-dir', MDGEN_CKPT_DIR,
    ], check=True)
    print(f'Downloaded: {MDGEN_CKPT}')
else:
    print(f'MDGen checkpoint exists: {MDGEN_CKPT}')


In [ ]:
import sys, os

mdgen_script = 'extern/mdgen/sim_inference.py'
mdgen_out = f'{DRIVE_BASE}/outputs/mdgen_baseline'
os.makedirs(mdgen_out, exist_ok=True)

# Run MDGen on training protein
print(f'Running MDGen on {PROTEIN} ...')
!python {mdgen_script} \
    --sim_ckpt {MDGEN_CKPT} \
    --data_dir {DATA_DIR} \
    --split gen_model/splits/atlas.csv \
    --pdb_id {PROTEIN} \
    --num_frames 250 --num_rollouts 1 \
    --suffix _R1_latent \
    --xtc \
    --out_dir {mdgen_out}

# Run MDGen on eval protein
print(f'\nRunning MDGen on {EVAL_PROTEIN} ...')
!python {mdgen_script} \
    --sim_ckpt {MDGEN_CKPT} \
    --data_dir {DATA_DIR} \
    --split gen_model/splits/atlas.csv \
    --pdb_id {EVAL_PROTEIN} \
    --num_frames 250 --num_rollouts 1 \
    --suffix _R1_latent \
    --xtc \
    --out_dir {mdgen_out}


In [ ]:
# Evaluate MDGen baseline on both proteins
for prot in [PROTEIN, EVAL_PROTEIN]:
    mdgen_pdb = f'{mdgen_out}/{prot}.pdb'
    if os.path.exists(mdgen_pdb):
        print(f'\n{"="*60}')
        print(f'MDGen baseline evaluation: {prot}')
        print(f'{"="*60}')
        ref_parent = f'{DRIVE_BASE}/outputs/eval/mdgen_baseline/{prot}/ref/{prot}'
        os.makedirs(ref_parent, exist_ok=True)
        !python -c "
import numpy as np, sys; sys.path.insert(0,'extern/mdgen')
from mdgen.utils import atom14_to_pdb
from mdgen.residue_constants import restype_order
import pandas as pd, mdtraj
seq_df = pd.read_csv('gen_model/splits/atlas.csv', index_col='name')
seqres = seq_df.loc['{prot}', 'seqres']
aatype = np.array([restype_order[c] for c in seqres])
refs = [np.load(f'{DATA_DIR}/{prot}/{prot}_R{{r}}_latent.npy').astype(np.float32) for r in [1,2,3]]
ref = np.concatenate(refs)
atom14_to_pdb(ref, aatype, '{ref_parent}/{prot}.pdb')
t = mdtraj.load('{ref_parent}/{prot}.pdb'); t.superpose(t)
t[0].save('{ref_parent}/{prot}.pdb'); t.save('{ref_parent}/{prot}.xtc')
"
        !PYTHONPATH=. python extern/mdgen/scripts/analyze_peptide_sim.py \
            --mddir outputs/eval/mdgen_baseline/{prot}/ref \
            --pdbdir {mdgen_out} \
            --pdb_id {prot} \
            --save --save_name mdgen_eval.pkl \
            --no_msm --plot
    else:
        print(f'MDGen output not found for {prot}')

# Display evaluation PDFs
import glob as _glob
from IPython.display import display, IFrame
for _pdf in _glob.glob(f'{DRIVE_BASE}/outputs/**/eval*/**/*.pdf', recursive=True)[-3:]:
    if _pdf not in globals().get('_shown_pdfs', set()):
        print(f'\n{_pdf}:')
        display(IFrame(_pdf, width=800, height=600))
        _shown_pdfs = globals().get('_shown_pdfs', set()) | {_pdf}


for _prot in [PROTEIN, EVAL_PROTEIN]:

for _prot in [PROTEIN, EVAL_PROTEIN]:
    viz_trajectory_pdb(f'{mdgen_out}/{_prot}.pdb',
                       f'{_prot} (MDGen pre-trained)')


## Step 12: Naive Single-Trajectory MDGen Baseline

Train MDGen's own architecture on the **same single trajectory** as our
model, using MDGen's default training — no SinFusion anti-overfitting.

| Technique | Applied? | Notes |
|-----------|----------|-------|
| Batch size = 1 | Yes | Already MDGen ATLAS default |
| Gradient clipping = 1.0 | Yes | Already MDGen default |
| EMA | No | Not enabled by default |
| Virtual epoch | No | Uses raw dataset size |
| Spatial crop | No | Fixed `--crop 256` (max length, not random 3D crop) |
| Stratified noise | No | Uniform t sampling |
| Curriculum delta_t | No | Fixed frame windows |

This is the **lower bound** for single-trajectory training: same data as
our model, but none of the anti-overfitting techniques.

In [ ]:
import os, subprocess

# Create a single-protein split CSV for MDGen training
import pandas as pd

atlas_df = pd.read_csv('gen_model/splits/atlas.csv', index_col='name')
single_train = atlas_df.loc[[PROTEIN]]
single_val = atlas_df.loc[[PROTEIN]]  # overfit check: val = train

os.makedirs(f'{DRIVE_BASE}/outputs/mdgen_naive', exist_ok=True)
train_split = f'{DRIVE_BASE}/outputs/mdgen_naive/{PROTEIN}_train.csv'
val_split   = f'{DRIVE_BASE}/outputs/mdgen_naive/{PROTEIN}_val.csv'
single_train.to_csv(train_split)
single_val.to_csv(val_split)
print(f'Created single-protein splits: {train_split}')


In [ ]:
if RUN_MDGEN_TRAIN:
    import os
    
    MDGEN_NAIVE_DIR = SAVE_DIR.replace('conditional/', 'mdgen_naive/')
    os.makedirs(MDGEN_NAIVE_DIR, exist_ok=True)
    os.environ['MODEL_DIR'] = MDGEN_NAIVE_DIR
    
    # Train MDGen on single protein with its ATLAS config
    # (num_frames=250, batch_size=1, prepend_ipa, crop up to 256 residues)
    # Reduced epochs since we're training on ~1/1000th of the data
    MDGEN_EPOCHS = 100     # 100 for Colab; 500 for full training
    
    !PYTHONPATH=. python extern/mdgen/train.py \
        --sim_condition \
        --train_split {train_split} \
        --val_split {val_split} \
        --data_dir {DATA_DIR} \
        --num_frames 250 \
        --batch_size 1 \
        --prepend_ipa \
        --crop 256 \
        --val_repeat 1 \
        --epochs {MDGEN_EPOCHS} \
        --atlas \
        --ckpt_freq 50 \
        --suffix _R{REPLICA}_latent \
        --no_validate \
        --run_name mdgen_naive_{PROTEIN}


In [ ]:
if RUN_MDGEN_TRAIN:
    import glob
    
    # Find latest checkpoint
    naive_ckpts = sorted(glob.glob(f'{MDGEN_NAIVE_DIR}/*.ckpt'))
    naive_ckpt = naive_ckpts[-1] if naive_ckpts else None
    
    if naive_ckpt:
        print(f'Naive MDGen checkpoint: {naive_ckpt}')
        mdgen_naive_out = f'{DRIVE_BASE}/outputs/mdgen_naive/inference'
        os.makedirs(mdgen_naive_out, exist_ok=True)
    
        # Run on training protein
        !PYTHONPATH=. python extern/mdgen/sim_inference.py \
            --sim_ckpt {naive_ckpt} \
            --data_dir {DATA_DIR} \
            --split gen_model/splits/atlas.csv \
            --pdb_id {PROTEIN} \
            --num_frames 250 --num_rollouts 1 \
            --suffix _R1_latent \
            --xtc \
            --out_dir {mdgen_naive_out}
    
        # Run on eval protein
        !PYTHONPATH=. python extern/mdgen/sim_inference.py \
            --sim_ckpt {naive_ckpt} \
            --data_dir {DATA_DIR} \
            --split gen_model/splits/atlas.csv \
            --pdb_id {EVAL_PROTEIN} \
            --num_frames 250 --num_rollouts 1 \
            --suffix _R1_latent \
            --xtc \
            --out_dir {mdgen_naive_out}
    
        # Evaluate
        for prot in [PROTEIN, EVAL_PROTEIN]:
            pdb = f'{mdgen_naive_out}/{prot}.pdb'
            if os.path.exists(pdb):
                print(f'\nEvaluating naive MDGen on {prot} ...')
                ref_dir = f'{DRIVE_BASE}/outputs/eval/mdgen_naive/{prot}/ref/{prot}'
                os.makedirs(ref_dir, exist_ok=True)
                !python -c "
    import numpy as np, sys; sys.path.insert(0,'extern/mdgen')
    from mdgen.utils import atom14_to_pdb
    from mdgen.residue_constants import restype_order
    import pandas as pd, mdtraj
    seq_df = pd.read_csv('gen_model/splits/atlas.csv', index_col='name')
    seqres = seq_df.loc['{prot}', 'seqres']
    aatype = np.array([restype_order[c] for c in seqres])
    refs = [np.load(f'{DATA_DIR}/{prot}/{prot}_R{{r}}_latent.npy').astype(np.float32) for r in [1,2,3]]
    ref = np.concatenate(refs)
    atom14_to_pdb(ref, aatype, '{ref_dir}/{prot}.pdb')
    t = mdtraj.load('{ref_dir}/{prot}.pdb'); t.superpose(t)
    t[0].save('{ref_dir}/{prot}.pdb'); t.save('{ref_dir}/{prot}.xtc')
    "
                !PYTHONPATH=. python extern/mdgen/scripts/analyze_peptide_sim.py \
                    --mddir outputs/eval/mdgen_naive/{prot}/ref \
                    --pdbdir {mdgen_naive_out} \
                    --pdb_id {prot} \
                    --save --save_name mdgen_naive_eval.pkl \
                    --no_msm --plot
    else:
        print('No naive MDGen checkpoint found. Training may have failed.')
    
    # Display evaluation PDFs
    import glob as _glob
    from IPython.display import display, IFrame
    for _pdf in _glob.glob(f'{DRIVE_BASE}/outputs/**/eval*/**/*.pdf', recursive=True)[-3:]:
        if _pdf not in globals().get('_shown_pdfs', set()):
            print(f'\n{_pdf}:')
            display(IFrame(_pdf, width=800, height=600))
            _shown_pdfs = globals().get('_shown_pdfs', set()) | {_pdf}
    
    
    for _prot in [PROTEIN, EVAL_PROTEIN]:
    
    for _prot in [PROTEIN, EVAL_PROTEIN]:
        viz_trajectory_pdb(f'{mdgen_naive_out}/{_prot}.pdb',
                           f'{_prot} (MDGen naive)')


## Step 13: MDGen + SinFusion Tricks (Enhanced Single-Trajectory)

Train MDGen's architecture on one trajectory with SinFusion anti-overfitting
applied via `gen_model/train_mdgen_enhanced.py`:

| Technique | Applied | How |
|-----------|---------|-----|
| Virtual epoch = 5000 | Yes | VirtualEpochDataset wrapper |
| Batch size = 1 | Yes | Already MDGen ATLAS default |
| Gradient clipping = 1.0 | Yes | Already MDGen default |
| EMA (0.999) | Yes | Already in MDGen |
| **Spatial crop (0.95)** | **Yes** | 3D CA-distance crop on loss mask |
| **Gaussian spatial bias** | **Yes** | Monkey-patched on mha_l (same sigma as our model) |
| **Stratified noise** | **Yes** | StratifiedTransportWrapper |
| Curriculum for delta_t | No | MDGen uses fixed windows, not variable stride |

**5-way comparison:**
- **MDGen naive** (Step 12): MDGen arch, no tricks
- **MDGen enhanced** (this step): MDGen arch + SinFusion tricks
- **Our model** (Steps 5-8): STAR-MD arch + SinFusion tricks

If enhanced >> naive: SinFusion protocol is the key.
If our model >> enhanced: STAR-MD architecture matters too.

In [ ]:
if RUN_MDGEN_TRAIN:
    MDGEN_ENHANCED_DIR = SAVE_DIR.replace('conditional/', 'mdgen_enhanced/')
    MDGEN_EPOCHS = 100     # 100 for Colab; 500 for full training
    
    !PYTHONPATH=. python gen_model/train_mdgen_enhanced.py \
        --protein {PROTEIN} \
        --replica {REPLICA} \
        --data_dir {DATA_DIR} \
        --epochs {MDGEN_EPOCHS} \
        --crop_ratio 0.95 \
        --virtual_epoch_size 5000 \
        --save_dir checkpoints/mdgen_enhanced


In [ ]:
if RUN_MDGEN_TRAIN:
    import glob, os
    
    enhanced_ckpts = sorted(glob.glob(f'{MDGEN_ENHANCED_DIR}/*.ckpt'))
    enhanced_ckpt = enhanced_ckpts[-1] if enhanced_ckpts else None
    
    if enhanced_ckpt:
        print(f'Enhanced MDGen checkpoint: {enhanced_ckpt}')
        mdgen_enh_out = f'{DRIVE_BASE}/outputs/mdgen_enhanced/inference'
        os.makedirs(mdgen_enh_out, exist_ok=True)
    
        for prot in [PROTEIN, EVAL_PROTEIN]:
            print(f'\nRunning enhanced MDGen on {prot} ...')
            !PYTHONPATH=. python extern/mdgen/sim_inference.py \
                --sim_ckpt {enhanced_ckpt} \
                --data_dir {DATA_DIR} \
                --split gen_model/splits/atlas.csv \
                --pdb_id {prot} \
                --num_frames 250 --num_rollouts 1 \
                --suffix _R1_latent \
                --xtc \
                --out_dir {mdgen_enh_out}
    
        # Evaluate
        for prot in [PROTEIN, EVAL_PROTEIN]:
            pdb = f'{mdgen_enh_out}/{prot}.pdb'
            if os.path.exists(pdb):
                print(f'\nEvaluating enhanced MDGen on {prot} ...')
                ref_dir = f'{DRIVE_BASE}/outputs/eval/mdgen_enhanced/{prot}/ref/{prot}'
                os.makedirs(ref_dir, exist_ok=True)
                !python -c "
    import numpy as np, sys; sys.path.insert(0,'extern/mdgen')
    from mdgen.utils import atom14_to_pdb
    from mdgen.residue_constants import restype_order
    import pandas as pd, mdtraj
    seq_df = pd.read_csv('gen_model/splits/atlas.csv', index_col='name')
    seqres = seq_df.loc['{prot}', 'seqres']
    aatype = np.array([restype_order[c] for c in seqres])
    refs = [np.load(f'{DATA_DIR}/{prot}/{prot}_R{{r}}_latent.npy').astype(np.float32) for r in [1,2,3]]
    ref = np.concatenate(refs)
    atom14_to_pdb(ref, aatype, '{ref_dir}/{prot}.pdb')
    t = mdtraj.load('{ref_dir}/{prot}.pdb'); t.superpose(t)
    t[0].save('{ref_dir}/{prot}.pdb'); t.save('{ref_dir}/{prot}.xtc')
    "
                !PYTHONPATH=. python extern/mdgen/scripts/analyze_peptide_sim.py \
                    --mddir outputs/eval/mdgen_enhanced/{prot}/ref \
                    --pdbdir {mdgen_enh_out} \
                    --pdb_id {prot} \
                    --save --save_name mdgen_enhanced_eval.pkl \
                    --no_msm --plot
    else:
        print('No enhanced MDGen checkpoint found.')
    
    # Display evaluation PDFs
    import glob as _glob
    from IPython.display import display, IFrame
    for _pdf in _glob.glob(f'{DRIVE_BASE}/outputs/**/eval*/**/*.pdf', recursive=True)[-3:]:
        if _pdf not in globals().get('_shown_pdfs', set()):
            print(f'\n{_pdf}:')
            display(IFrame(_pdf, width=800, height=600))
            _shown_pdfs = globals().get('_shown_pdfs', set()) | {_pdf}
    
    
    for _prot in [PROTEIN, EVAL_PROTEIN]:
    
    for _prot in [PROTEIN, EVAL_PROTEIN]:
        viz_trajectory_pdb(f'{mdgen_enh_out}/{_prot}.pdb',
                           f'{_prot} (MDGen enhanced)')
